In [ ]:
# ==============================================================================
# Model B Training Pipeline: Kaggle GPU + W&B
# Dataset: mohdahsan8178/sih-model-b-dataset
# Paste this ENTIRE cell into a Kaggle notebook. Safe for "Save & Run All".
# ==============================================================================

# ---- 0. Dependencies (safe to re-run; quiet) ----
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "wandb", "albumentations>=2.0.0"], check=False)

import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             confusion_matrix, classification_report)
import timm
import wandb


# ---- Temperature scaling (was missing from the recipe script) ----
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, val_logits, val_labels):
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)
        criterion = nn.CrossEntropyLoss()

        def eval_loss():
            optimizer.zero_grad()
            loss = criterion(self.forward(val_logits), val_labels)
            loss.backward()
            return loss

        optimizer.step(eval_loss)
        return float(self.temperature.item())


# ---- 1. W&B auth (never blocks in batch mode) ----
WANDB_MODE = "online"
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
except Exception as e:
    print("W&B secret unavailable (%s). Falling back to offline logging." % e)
    os.environ["WANDB_MODE"] = "offline"
    WANDB_MODE = "offline"

wandb.init(
    project="sih-model-b-trap-pest-classifier",
    mode=WANDB_MODE,
    config={
        "architecture": "mobilenetv3_small_100",
        "batch_size": 128,
        "lr": 1e-3,
        "epochs": 25,
        "loss": "ClassWeightedCrossEntropy",
        "weights": [0.534, 1.290, 1.176],
    },
)

# ---- 2. Data ----
WORK_DIR = Path("/kaggle/working")

print("--- DIRECTORY AUDIT ---")
print("CWD:", os.getcwd())
if Path("/kaggle/input").exists():
    print("/kaggle/input entries:")
    for item in Path("/kaggle/input").iterdir():
        print("  -", item.name, "(dir)" if item.is_dir() else "(file)")
        if item.is_dir():
            try:
                for sub in list(item.iterdir())[:8]:
                    print("      *", sub.name)
            except Exception as e:
                print("      * error listing:", e)

# 1. Download or locate dataset via kagglehub (recommended by Kaggle)
kh_dir = None
try:
    import kagglehub
    print("\nQuerying kagglehub for mohdahsan8178/sih-model-b-dataset...")
    kh_dir = Path(kagglehub.dataset_download("mohdahsan8178/sih-model-b-dataset"))
    print("kagglehub returned path: %s" % kh_dir)
except Exception as e:
    print("kagglehub lookup failed (%s). Proceeding with filesystem scan." % e)

# 2. Search for manifest across all possible locations
search_roots = []
if kh_dir and kh_dir.exists():
    search_roots.append(kh_dir)
if Path("/kaggle/input").exists():
    search_roots.append(Path("/kaggle/input"))
search_roots.extend([WORK_DIR, Path(".")])

manifest_path = None
for root in search_roots:
    matches = list(root.glob("**/model_b_manifest_v4.csv"))
    if matches:
        manifest_path = matches[0]
        print("Found manifest at: %s" % manifest_path)
        break

if manifest_path is None:
    raise FileNotFoundError("Could not find model_b_manifest_v4.csv! Checked roots: %s" % search_roots)

df_master = pd.read_csv(manifest_path)
print("Loaded manifest (%d rows)" % len(df_master))
print(pd.crosstab(df_master["class_name"], df_master["split"], margins=True))

# 3. Locate image patch root directory
sample_crop = "small_pale_winged_000000.png"
sample_class = "small_pale_winged"
image_root = None

for root in [manifest_path.parent, manifest_path.parent.parent] + search_roots:
    for candidate in [
        root,
        root / "model_b_patches_v4",
        root / "model_b_patches_v4" / "model_b_patches_v4",
    ]:
        if (candidate / sample_class / sample_crop).exists() or (candidate / "model_b_patches_v4" / sample_class / sample_crop).exists():
            image_root = candidate
            break
    if image_root:
        break

if image_root is None:
    for root in search_roots:
        found_imgs = list(root.glob("**/" + sample_crop))
        if found_imgs:
            p = found_imgs[0].parent
            image_root = p.parent if p.name == sample_class else p
            break

if image_root is None:
    raise FileNotFoundError("Could not locate patch images anywhere under %s" % search_roots)

IMAGE_ROOT = image_root
print("Resolved IMAGE_ROOT: %s" % IMAGE_ROOT)


class StickyTrapDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = self.root_dir / row["rel_path"]
        if not p.exists():
            p = self.root_dir / row["class_name"] / ("%s.png" % row["crop_id"])
        if not p.exists():
            p = self.root_dir / "model_b_patches_v4" / row["class_name"] / ("%s.png" % row["crop_id"])
        im = Image.open(p).convert("RGB")
        im_np = np.array(im)
        if self.transform:
            tensor = self.transform(image=im_np)["image"]
        else:
            tensor = torch.from_numpy(im_np.transpose(2, 0, 1)).float() / 255.0
        return tensor, int(row["class_idx"])


# ---- 3. Augmentations (ESP32-CAM degradation simulation) ----
def build_train_transform():
    """Tolerates albumentations 1.x and 2.x parameter naming."""
    try:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Downscale(scale_range=(0.25, 0.60), p=0.60),
            A.ImageCompression(quality_range=(15, 60), p=0.70),
            A.OneOf([
                A.MotionBlur(blur_limit=(3, 7)),
                A.Defocus(radius=(1, 3)),
                A.GaussianBlur(blur_limit=(3, 5)),
            ], p=0.50),
            A.ISONoise(p=0.40),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                                 val_shift_limit=20, p=0.50),
            A.CoarseDropout(num_holes_range=(1, 4),
                            hole_height_range=(4, 12),
                            hole_width_range=(4, 12),
                            fill=255, p=0.30),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    except (TypeError, ValueError) as e:
        print("New albumentations API unavailable (%s); using legacy names." % e)
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Downscale(scale_min=0.25, scale_max=0.60, p=0.60),
            A.ImageCompression(quality_lower=15, quality_upper=60, p=0.70),
            A.OneOf([
                A.MotionBlur(blur_limit=(3, 7)),
                A.GaussianBlur(blur_limit=(3, 5)),
            ], p=0.50),
            A.ISONoise(p=0.40),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                                 val_shift_limit=20, p=0.50),
            A.CoarseDropout(max_holes=4, max_height=12, max_width=12,
                            fill_value=255, p=0.30),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])


train_transform = build_train_transform()
eval_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

df_train = df_master[df_master["split"] == "train"]
df_val = df_master[df_master["split"] == "val"]
df_test_indist = df_master[df_master["split"] == "test_indist"]
df_test_cross = df_master[df_master["split"] == "test_cross_card"]
df_ood = df_master[df_master["split"] == "openset_eval"]

train_loader = DataLoader(StickyTrapDataset(df_train, IMAGE_ROOT, train_transform),
                          batch_size=128, shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=False)
val_loader = DataLoader(StickyTrapDataset(df_val, IMAGE_ROOT, eval_transform),
                        batch_size=128, shuffle=False, num_workers=2)
test_indist_loader = DataLoader(StickyTrapDataset(df_test_indist, IMAGE_ROOT, eval_transform),
                                batch_size=128, shuffle=False, num_workers=2)
test_cross_loader = DataLoader(StickyTrapDataset(df_test_cross, IMAGE_ROOT, eval_transform),
                               batch_size=128, shuffle=False, num_workers=2)
ood_loader = DataLoader(StickyTrapDataset(df_ood, IMAGE_ROOT, eval_transform),
                        batch_size=128, shuffle=False, num_workers=2)

# ---- 4. Model ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: %s" % device)

model = timm.create_model("mobilenetv3_small_100", pretrained=True, num_classes=3).to(device)
class_weights = torch.tensor([0.534, 1.290, 1.176], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-5)

CLASS_NAMES = ["small_pale_winged", "larger_insect", "debris"]


def run_eval(loader, net):
    net.eval()
    preds, targets, logits = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = net(x)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            targets.extend(y.numpy())
            logits.append(out.cpu().numpy())
    return (np.array(targets), np.array(preds),
            np.concatenate(logits, axis=0))


# ---- 5. Training loop with early stopping ----
best_val_f1 = 0.0
best_model_path = WORK_DIR / "best_model_b.pt"
patience, epochs_no_improve = 5, 0

for epoch in range(1, 26):
    model.train()
    total_loss, total_count = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        total_count += len(y)
    scheduler.step()

    y_val, p_val, _ = run_eval(val_loader, model)
    val_f1 = f1_score(y_val, p_val, average="macro")
    val_bal = balanced_accuracy_score(y_val, p_val)

    print("Epoch %02d | Train Loss: %.4f | Val Macro-F1: %.4f | Bal-Acc: %.2f%%"
          % (epoch, total_loss / total_count, val_f1, val_bal * 100))
    wandb.log({
        "epoch": epoch,
        "train_loss": total_loss / total_count,
        "val_macro_f1": val_f1,
        "val_balanced_acc": val_bal,
        "learning_rate": scheduler.get_last_lr()[0],
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save({"model_state": model.state_dict(), "val_f1": float(val_f1)},
                   best_model_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping at epoch %d (no val improvement for %d epochs)."
                  % (epoch, patience))
            break

# ---- 6. Reload best checkpoint ----
ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()
print("\nLoaded best checkpoint (val Macro-F1 = %.4f)" % ckpt["val_f1"])

# ---- 7. Benchmarks ----
y_in, p_in, _ = run_eval(test_indist_loader, model)
print("\n" + "=" * 62)
print("IN-DISTRIBUTION TEST (group-isolated cards)")
print("=" * 62)
print("Macro-F1           : %.4f" % f1_score(y_in, p_in, average="macro"))
print("Balanced Accuracy  : %.2f%%" % (balanced_accuracy_score(y_in, p_in) * 100))
print(classification_report(y_in, p_in, target_names=CLASS_NAMES, digits=4))

y_cr, p_cr, _ = run_eval(test_cross_loader, model)
print("\n" + "=" * 62)
print("FINAL CROSS-CARD BENCHMARK (84 HELD-OUT 4TU CARDS)  <-- HEADLINE")
print("=" * 62)
print("Macro-F1 (HEADLINE): %.4f" % f1_score(y_cr, p_cr, average="macro"))
print("Balanced Accuracy  : %.2f%%" % (balanced_accuracy_score(y_cr, p_cr) * 100))
print("\nConfusion Matrix (rows = true, cols = predicted):")
print("             %-18s %-15s %-10s" % tuple(CLASS_NAMES))
cm = confusion_matrix(y_cr, p_cr)
for i, name in enumerate(CLASS_NAMES):
    print("%-18s %-18d %-15d %-10d" % (name, cm[i][0], cm[i][1], cm[i][2]))
print("\n" + classification_report(y_cr, p_cr, target_names=CLASS_NAMES, digits=4))

wf_total = int((y_cr == 0).sum())
wf_mis_larger = int(((y_cr == 0) & (p_cr == 1)).sum())
leak_rate = (wf_mis_larger / wf_total) * 100.0
verdict = "PASS" if leak_rate <= 5.0 else "FAIL"
print("Operational ETL Leakage (Whitefly -> Larger): %.2f%%  [%s, ceiling 5.0%%]"
      % (leak_rate, verdict))

li_recall = cm[1][1] / cm[1].sum() * 100.0
print("larger_insect recall (the morphology-bearing number): %.2f%%" % li_recall)

# ---- 8. Calibration: temperature + energy OOD threshold ----
# Recomputed from the BEST checkpoint, not the last epoch.
y_valb, p_valb, l_valb = run_eval(val_loader, model)

temp_scaler = TemperatureScaler().to(device)
T_cal = temp_scaler.fit(
    torch.tensor(l_valb, dtype=torch.float32).to(device),
    torch.tensor(y_valb, dtype=torch.long).to(device),
)
print("\nFitted Temperature T_cal      = %.4f" % T_cal)

from scipy.special import logsumexp
id_energy = -T_cal * logsumexp(l_valb / T_cal, axis=1)
tau_energy = float(np.percentile(id_energy, 95.0))
print("Fitted Energy Threshold       = %.4f  (95%% ID TPR)" % tau_energy)

_, _, l_ood = run_eval(ood_loader, model)
ood_energy = -T_cal * logsumexp(l_ood / T_cal, axis=1)
ood_rejection_rate = float((ood_energy > tau_energy).mean()) * 100.0
print("OOD Rejection on hard non-targets: %.2f%%" % ood_rejection_rate)

wandb.log({
    "cross_card_macro_f1": f1_score(y_cr, p_cr, average="macro"),
    "cross_card_balanced_acc": balanced_accuracy_score(y_cr, p_cr),
    "etl_leakage_rate_pct": leak_rate,
    "larger_insect_recall_pct": li_recall,
    "T_cal": T_cal,
    "tau_energy": tau_energy,
    "ood_rejection_rate_pct": ood_rejection_rate,
})

# ---- 9. ONNX export ----
dummy_input = torch.randn(1, 3, 64, 64).to(device)
onnx_path = WORK_DIR / "model_b_calibrated.onnx"
torch.onnx.export(
    model, dummy_input, str(onnx_path),
    opset_version=13,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
)
print("\nExported ONNX -> %s" % onnx_path)

# ---- 10. Calibration constants for the edge config ----
print("\n" + "=" * 62)
print("COPY THESE INTO configs/train_config.py")
print("=" * 62)
print("MODEL_B_T_CAL      = %.4f" % T_cal)
print("MODEL_B_TAU_ENERGY = %.4f" % tau_energy)
print("MODEL_B_TAU_CONF   = 0.60")
print("=" * 62)

wandb.finish()
print("\nDone. Download model_b_calibrated.onnx from the Output pane.")
